In [15]:
import pandas as pd
import numpy as np

In [16]:
folder = r"D:\projects\Brazilian E-Commerce Public Dataset\data\sampled data\\"

In [19]:
order_level = pd.read_csv(folder + "order_level.csv")
item_level = pd.read_csv(folder + "item_level.csv")

orders_s = pd.read_csv(folder + "orders.csv")
customers_s   = pd.read_csv(folder + "customers.csv")
order_items_s = pd.read_csv(folder + "order_items.csv")
products_s    = pd.read_csv(folder + "products.csv")
sellers_s     = pd.read_csv(folder + "sellers.csv")
print("Loaded")
print("order_level:", order_level.shape, "| item_level:", item_level.shape)


Loaded
order_level: (10000, 23) | item_level: (11383, 22)


\\Missing_value_analysis

In [20]:

def missing_report(df, table_name):
    
    total = len(df)
    miss = df.isnull().sum()                    
    pct  = (miss / total * 100).round(2)         
    report = pd.DataFrame({
        "missing_count": miss,
        "missing_pct": pct
    })

    report = report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)
    print(f"\n===== {table_name} (total {total:,} rows) =====")
    if report.empty:
        print("No missing values ")
    else:
        print(report)
    return report

miss_order = missing_report(order_level, "order_level")
miss_item  = missing_report(item_level, "item_level")


===== order_level (total 10,000 rows) =====
                               missing_count  missing_pct
order_delivered_customer_date            285         2.85
order_delivered_carrier_date             179         1.79
n_reviews                                 88         0.88
avg_review_score                          88         0.88
geolocation_lat                           35         0.35
geolocation_zip_code_prefix               35         0.35
geolocation_city                          35         0.35
gelocation_lng                            35         0.35
geolocation_state                         35         0.35
order_approved_at                         24         0.24
n_payments_methods                         1         0.01
max_installments                           1         0.01
total_payment_value                        1         0.01
main_payment_type                          1         0.01

===== item_level (total 11,383 rows) =====
                               missing_co

\\Duplicate Analysis

In [25]:
def duplicate_report(df, table_name, key_col=None):
    full_dup = df.duplicated().sum()
    print(f"\n===={table_name} ====")
    print(f"Fully fuplicate rows: {full_dup}")
    if key_col:
        key_dup = df.duplicated(subset=[key_col]).sum()
        print(f"Duplicate '{key_col}':{key_dup}")
    return full_dup

duplicate_report(order_level, "order_level", key_col="order_id")
duplicate_report(item_level, "item_level")
duplicate_report(customers_s, "customers", key_col="customer_id")
    


====order_level ====
Fully fuplicate rows: 0
Duplicate 'order_id':0

====item_level ====
Fully fuplicate rows: 0

====customers ====
Fully fuplicate rows: 0
Duplicate 'customer_id':0


np.int64(0)

\\Check orphans

In [31]:
def check_orphans(child_df, child_key, parent_df, parent_key, label):
    child_keys = set(child_df[child_key].dropna())
    parent_keys = set(parent_df[parent_key])
    orphans = child_keys - parent_keys
    print(f"{label}: {len(orphans)} orphan keys", "YES" if len(orphans)!=0 else "NO")
    return orphans

print("==== Referential Integrity ====")  
check_orphans(orders_s, "customer_id", customers_s, "customer_id",
              "orders.customer_id -> customers")
check_orphans(order_items_s, "order_id", orders_s, "order_id",
              "order_items.order_id -> orders")
check_orphans(order_items_s, "product_id", products_s, "product_id",
              "order_items.product_id -> products")
check_orphans(order_items_s, "seller_id", sellers_s, "seller_id",
              "order_items.seller_is -> sellers")

==== Referential Integrity ====
orders.customer_id -> customers: 0 orphan keys NO
order_items.order_id -> orders: 0 orphan keys NO
order_items.product_id -> products: 0 orphan keys NO
order_items.seller_is -> sellers: 0 orphan keys NO


set()

\\Invalid Value + Consistency Checks

In [37]:
print("==== Invalid Value Checks ====")

neg_price = (item_level["price"] <= 0).sum()
neg_freight = (item_level["freight_value"] < 0).sum()
print(f"Price <= 0     :   {neg_price}")
print(f"Freight < 0    :   {neg_freight}")

Invalid_review = (~order_level["avg_review_score"].between(1,5)).sum() if "avg_review_score" in order_level else 0
print(f"Review out of 1-5: {Invalid_review} (Ignore NaN)")

od = order_level.copy()
od["order_purchase_timestamp"] = pd.to_datetime(od["order_purchase_timestamp"], errors="coerce")
od["order_delivered_customer_date"] = pd.to_datetime(od["order_delivered_customer_date"], errors="coerce")
Invalid_delivery = (od["order_delivered_customer_date"] < od["order_purchase_timestamp"]).sum()
print(f"Delivered BEFORE purchase: {Invalid_delivery}")

zero_inst = (order_level["max_installments"] == 0).sum() if "max_installments" in order_level else 0
print(f"installments = 0 : {zero_inst}")

==== Invalid Value Checks ====
Price <= 0     :   0
Freight < 0    :   0
Review out of 1-5: 88 (Ignore NaN)
Delivered BEFORE purchase: 0
installments = 0 : 0


\\Overall_Summary

In [39]:
def completeness_score(df, name):
    total_cells = df.shape[0] * df.shape[1]
    filled = total_cells - df.isnull().sum().sum()
    score = round(filled / total_cells * 100, 2)
    print(f"{name:13s} completeness: {score}% ({filled:,}/{total_cells:,} cells filled)")
    return score

print("==== Completeness score ====")
completeness_score(order_level, "order_level")
completeness_score(item_level, "item_level")

==== Completeness score ====
order_level   completeness: 99.63% (229,157/230,000 cells filled)
item_level    completeness: 99.7% (249,668/250,426 cells filled)


np.float64(99.7)

\\Data Cleaning

In [41]:
date_cols_order= [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
        ]

for col in date_cols_order:
    order_level[col] = pd.to_datetime(order_level[col], errors ="coerce")
    
item_level["order_purchase_timestamp"] = pd.to_datetime(
    item_level["order_purchase_timestamp"], errors="coerce"
)

print(order_level[date_cols_order].dtypes)


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


\\Handling Missing Values

In [48]:

if "n_payments_methods" in order_level.columns:
    order_level = order_level.rename(columns={"n_payments_methods": "n_payment_methods"})
    
# 1. Review
order_level["has_review"] = order_level["avg_review_score"].notna().astype(int)
order_level["n_reviews"] = order_level["n_reviews"].fillna(0).astype(int)

# 2. Payment 
if "total_payment_value" in order_level.columns:
    order_level["total_payment_value"] = order_level["total_payment_value"].fillna(0)

if "max_installments" in order_level.columns:
    order_level["max_installments"] = order_level["max_installments"].fillna(1)

if "n_payment_methods" in order_level.columns:
    order_level["n_payment_methods"] = order_level["n_payment_methods"].fillna(0).astype(int)

if "main_payment_type" in order_level.columns:
    order_level["main_payment_type"] = order_level["main_payment_type"].fillna("unknown")

# 3. Delivery flag
order_level["is_delivered"] = order_level["order_delivered_customer_date"].notna().astype(int)

# 4. City/state
order_level["customer_city"] = order_level["customer_city"].fillna("unknown")
order_level["customer_state"] = order_level["customer_state"].fillna("unknown")

print("order_level remaining missing:")
print(order_level.isnull().sum()[order_level.isnull().sum() > 0])

order_level remaining missing:
order_approved_at                 24
order_delivered_carrier_date     179
order_delivered_customer_date    285
avg_review_score                  88
geolocation_zip_code_prefix       35
geolocation_lat                   35
gelocation_lng                    35
geolocation_city                  35
geolocation_state                 35
dtype: int64


In [51]:
#clean_item_level
item_level["product_category_name_english"] = (
    item_level["product_category_name_english"]
    .fillna(item_level["product_category_name"])  #First_try_portugese
    .fillna("unknown")  #otherwise_unknown
)

#Median_fill
for col in ["product_weight_g", "product_length_cm",
            "product_height_cm", "product_width_cm"]:
    median_val = item_level[col].median()
    item_level[col] = item_level[col].fillna(median_val)

for col in ["product_name_lenght", "product_description_lenght", "product_photos_qty"]:
    item_level[col] = item_level[col].fillna(0)

print("item_level remaining missing:")
print(item_level.isnull().sum()[item_level.isnull().sum() > 0])

    

item_level remaining missing:
product_category_name    151
dtype: int64


\\Fix_invalid_values + standardize formats

In [64]:
#Installments 0 to 1
if "max_installments" in order_level.columns:
    order_level.loc[order_level["max_installments"] == 0, "max_installments"] = 1

#Text_standardize
order_level["customer_city"] = order_level["customer_city"].str.lower().str.strip()
order_level["customer_state"] = order_level["customer_state"].str.lower().str.strip()
item_level["seller_city"] = item_level["seller_city"].str.lower().str.strip()
item_level["seller_state"] = item_level["seller_state"].str.lower().str.strip()

#Data_types_fix
order_level["max_installments"] = order_level["max_installments"].astype(int)

print("Invalid fixes done")
print("Sample cities:", order_level["customer_city"].unique()[:5])
print("Installments=0 now:", (order_level["max_installments"]==0).sum())









Invalid fixes done
Sample cities: <ArrowStringArray>
['sobradinho', 'belo horizonte', 'curitiba', 'rio de janeiro', 'londrina']
Length: 5, dtype: str
Installments=0 now: 0


In [62]:
#If any duplicates is there should remove before saving
before_o = len(order_level); order_level = order_level.drop_duplicates()
before_i = len(item_level); item_level = item_level.drop_duplicates()
print(f"order_level: {before_o} -> {len(order_level)} (removed {before_o-len(order_level)})")
print(f"item_level: {before_i} -> {len(item_level)} (removed {before_i-len(item_level)})")

#Saving_Clean_versions
folder = r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\\"
order_level.to_csv(folder + "order_level_clean.csv", index=False)
item_level.to_csv(folder + "item_level_clean.csv", index=False)
print("\nsaved: order_level_clean.csv, item_level_clean.csv")

order_level: 10000 -> 10000 (removed 0)
item_level: 11383 -> 11383 (removed 0)

saved: order_level_clean.csv, item_level_clean.csv


## Key Observations

- Delivery timestamps contribute the most missing values (2.85%) — structurally absent for undelivered orders, not data entry errors.
- Both tables are fully deduplicated and all four foreign key relationships (customer, order, product, seller) have zero orphan keys, confirming a clean sample extraction.
- No negative prices, no negative freight values, and no order delivered before its purchase date — numeric and temporal fields are logically consistent throughout.
- Both datasets exceed 99.6% completeness; remaining gaps are confined to delivery dates, review scores, and geolocation fields.
- 151 items in `item_level` carry no category name even in Portuguese — these fall back to `"unknown"` and may need manual mapping before category-level analysis.